# HANC with heterogeneous discount factors

**Table of contents**<a id='toc0_'></a>
- 1. [Setup](#toc1_)
- 2. [Steady state](#toc2_)
- 3. [MPC](#toc3_)
- 4. [Calibrate the dispersion of discount factors](#toc4_)
- 5. [Wealth distribution](#toc5_)

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

**Exercise 1:** the HANC model with three types of households, $\beta_i\in\{\beta-\delta^{\beta},\beta,\beta+\delta^{\beta}\}$, each one third of the population. Find $\delta^{\beta}$ such that the mean MPC is 0.27, while keeping $K/Y=16$.

What changed relative to `0_HANC`:

1. `HANCModel.py`: `par.Nfix = 3`, the parameters `par.beta_mean` and `par.beta_delta`, and `par.beta_grid` is allocated in `allocate()`.
1. `steady_state.py`: `prepare_hh_ss` fills `par.beta_grid` and gives each type one third of the initial distribution. `find_ss` is the $\beta$ method from the lecture: $\alpha$, $\Gamma$ and $\delta$ in closed form from the targets, then a root-finder on `par.beta_mean`.
1. `household_problem.py`: a loop over the fixed types, each with its own `par.beta_grid[i_fix]`.

## 1. <a id='toc1_'></a>[Setup](#toc0_)

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize

from HANCModel import HANCModelClass

plt.rcParams.update({"axes.grid" : True, "grid.color": "black", "grid.alpha":"0.25", "grid.linestyle": "--"})
plt.rcParams.update({'font.size': 14})

In [ ]:
model = HANCModelClass(name='baseline')
par = model.par
ss = model.ss

## 2. <a id='toc2_'></a>[Steady state](#toc0_)

Find the steady state with `model.find_ss()`, then look at `par.beta_mean` and `par.beta_grid`.

In [ ]:
# find the steady state and print par.beta_mean and par.beta_grid


The calibration targets, at the quarterly frequency:

- $Y=1$, a normalization, pins down $\Gamma$
- $K/Y=16$, an annual wealth-to-output ratio of 4, pins down $\beta$
- $r=0.05/4$, a 5% annual interest rate, pins down $\delta$
- $wL/Y=2/3$, the labor share, pins down $\alpha$

In [ ]:
print(f'Y = {ss.Y:.4f}')
print(f'K/Y = {ss.K/ss.Y:.4f}')
print(f'r = {ss.r:.4f}')
print(f'w*L/Y = {ss.w*ss.L/ss.Y:.4f}')
print(f'clearing_A = {ss.clearing_A:.2e}, clearing_Y = {ss.clearing_Y:.2e}')

In [ ]:
model.info(ss=True)

## 3. <a id='toc3_'></a>[MPC](#toc0_)

In [ ]:
def calc_MPC(par,ss):
    """ MPC out of cash-on-hand from the slope of the consumption function """

    MPC = np.zeros(ss.D.shape)
    dc = ss.c[:,:,1:]-ss.c[:,:,:-1]
    dm = (1+ss.r)*(par.a_grid[1:]-par.a_grid[:-1])
    MPC[:,:,:-1] = dc/dm
    MPC[:,:,-1] = MPC[:,:,-2]

    return MPC

In [ ]:
MPC = calc_MPC(par,ss)

print(f'mean MPC: {np.sum(MPC*ss.D):.3f}')
for i_fix in range(par.Nfix):
    MPC_type = np.sum(MPC[i_fix]*ss.D[i_fix])/np.sum(ss.D[i_fix])
    print(f'beta = {par.beta_grid[i_fix]:.3f}: mean MPC = {MPC_type:.3f}, share of wealth = {np.sum(ss.a[i_fix]*ss.D[i_fix])/ss.A_hh:.3f}')

In [ ]:
fig, axes = plt.subplots(1,par.Nfix,figsize=(16,4),sharey=True)
for i_fix, ax in enumerate(axes):
    ax.plot(par.a_grid,MPC[i_fix].T)
    ax.set_xlim(-0.1,10)
    ax.set_title(f'beta = {par.beta_grid[i_fix]:.3f}')
    ax.set_xlabel('wealth')
axes[0].set_ylabel('MPC')
axes[-1].legend([f'z = {z:.2f}' for z in par.z_grid],fontsize=9)
plt.show()

**Question:** Why do the impatient households have a high MPC? Who holds the wealth?

## 4. <a id='toc4_'></a>[Calibrate the dispersion of discount factors](#toc0_)

The mean MPC depends on how far apart the discount factors are. For each guess on `par.beta_delta` we must solve for the steady state again, so that $K/Y=16$ still holds: a root-finder around a root-finder.

In [ ]:
# write calib_obj(beta_delta,model): set par.beta_delta, find the steady state, return the mean MPC minus 0.27


In [ ]:
# copy the model and call optimize.brentq on calib_obj, with beta_delta between 0.01 and 0.1


In [ ]:
par_calib = model_calib.par
ss_calib = model_calib.ss

print(f'beta_delta = {par_calib.beta_delta:.4f}')
print(f'beta_grid = {par_calib.beta_grid}')
print(f'r = {ss_calib.r*100:.2f} %, K/Y = {ss_calib.K/ss_calib.Y:.2f}')

## 5. <a id='toc5_'></a>[Wealth distribution](#toc0_)

Compute the wealth shares and compare them to the data, as in the lecture notebook. How did the distribution of wealth change?

In [ ]:
def get_share(D,a_grid,perc_bottom=0.0,perc_top=1.0):
    """ share of total wealth held between two percentiles of the wealth distribution """

    cdf = D.cumsum()
    wealth_cdf = (a_grid*D).cumsum()
    tot_wealth = (a_grid*D).sum()

    wealth_bottom = 0.0 if perc_bottom == 0.0 else np.interp(perc_bottom,cdf,wealth_cdf)
    wealth_top = tot_wealth if perc_top == 1.0 else np.interp(perc_top,cdf,wealth_cdf)

    return (wealth_top-wealth_bottom)/tot_wealth

In [ ]:
# compute the wealth shares of the calibrated model and compare them to ../wealth_psz.csv
